# ACCT 3343: Collecting financial data with Yahoo Finance

In this notebook, you will use Python to pull live financial data for a fixed set of 50 public companies. You will organize the results in pandas dataframes, inspect them, and save them for analysis in later classes.

This notebook collects data only. It does **not** calculate Quantitative Value scores, rankings, or investment signals.


## 1. Install and import the packages

Run this cell first. It installs Yahoo Finance's Python package and loads the tools used in the notebook.


In [ ]:
!pip install -q yfinance

from pathlib import Path
import time

import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display

pd.set_option("display.max_columns", 100)


## 2. Our fixed 50-company universe

These 50 companies are the standard classroom universe for the semester. The teaching-sector labels make the list easier to browse.


In [ ]:
TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "ORCL",
    "CSCO",
    "IBM",
    "CRM",
    "ADBE",
    "CAT",
    "DE",
    "HON",
    "UNP",
    "UPS",
    "RTX",
    "GE",
    "AMZN",
    "HD",
    "MCD",
    "NKE",
    "SBUX",
    "LOW",
    "TJX",
    "WMT",
    "COST",
    "PG",
    "KO",
    "PEP",
    "MO",
    "JNJ",
    "UNH",
    "PFE",
    "MRK",
    "ABBV",
    "TMO",
    "ABT",
    "XOM",
    "CVX",
    "COP",
    "SLB",
    "EOG",
    "JPM",
    "BAC",
    "WFC",
    "GS",
    "MS",
    "BLK",
    "NEE",
    "DUK",
    "SO",
    "AEP"
]

TEACHING_SECTORS = [
    "Technology",
    "Technology",
    "Technology",
    "Technology",
    "Technology",
    "Technology",
    "Technology",
    "Technology",
    "Industrials",
    "Industrials",
    "Industrials",
    "Industrials",
    "Industrials",
    "Industrials",
    "Industrials",
    "Consumer Discretionary",
    "Consumer Discretionary",
    "Consumer Discretionary",
    "Consumer Discretionary",
    "Consumer Discretionary",
    "Consumer Discretionary",
    "Consumer Discretionary",
    "Consumer Staples",
    "Consumer Staples",
    "Consumer Staples",
    "Consumer Staples",
    "Consumer Staples",
    "Consumer Staples",
    "Healthcare",
    "Healthcare",
    "Healthcare",
    "Healthcare",
    "Healthcare",
    "Healthcare",
    "Healthcare",
    "Energy",
    "Energy",
    "Energy",
    "Energy",
    "Energy",
    "Financials",
    "Financials",
    "Financials",
    "Financials",
    "Financials",
    "Financials",
    "Utilities",
    "Utilities",
    "Utilities",
    "Utilities"
]

universe = pd.DataFrame({
    "ticker": TICKERS,
    "teaching_sector": TEACHING_SECTORS,
})

assert len(TICKERS) == 50
assert len(set(TICKERS)) == 50

print(f"Fixed classroom universe: {len(TICKERS)} companies")
display(universe)


## 3. What happens when we run the download?

**Ticker → Yahoo Finance → financial statements and market data → pandas dataframes**

A dataframe is a table with rows and columns. Here, each financial-data row represents one company at one fiscal-period end. A company normally appears in several rows because Yahoo provides multiple annual periods.

The fields were selected by reviewing the ACCT 3343 / *Quantitative Value* course materials and mapping course-relevant accounting items to data available through Yahoo Finance. Missing exact concepts remain missing, and near-equivalent concepts are kept separate.

> Yahoo generally provides only about five years of annual statement history. Therefore, this dataset cannot fully reproduce course measures that require eight years of accounting history without another data source.


## 4. Yahoo field mappings and helper functions

The next two cells contain the verified download and cleaning machinery. They are collapsed because you do not need to understand Yahoo's technical details for today's lesson. You can expand them if you are curious.


In [ ]:
STATEMENT_FIELDS = {
    "income_statement": {
        "total_revenue": ["Total Revenue", "Operating Revenue"],
        "cost_of_revenue": ["Cost Of Revenue", "Reconciled Cost Of Revenue"],
        "gross_profit": ["Gross Profit"],
        "ebit": ["EBIT"],
        "operating_income": ["Operating Income", "Total Operating Income As Reported"],
        "ebitda": ["EBITDA", "Normalized EBITDA"],
        "normalized_income": ["Normalized Income"],
        "pretax_income": ["Pretax Income"],
        "tax_provision": ["Tax Provision"],
        "tax_rate_for_calcs": ["Tax Rate For Calcs"],
        "net_income": ["Net Income"],
        "net_income_continuing_operations": [
            "Net Income Continuous Operations",
            "Net Income From Continuing Operation Net Minority Interest",
        ],
        "selling_general_and_administration": [
            "Selling General And Administration",
            "General And Administrative Expense",
        ],
        "interest_expense": ["Interest Expense", "Interest Expense Non Operating"],
        "basic_average_shares": ["Basic Average Shares"],
        "diluted_average_shares": ["Diluted Average Shares"],
        "basic_eps": ["Basic EPS"],
        "diluted_eps": ["Diluted EPS"],
    },
    "balance_sheet": {
        "total_assets": ["Total Assets"],
        "current_assets": ["Current Assets", "Total Current Assets"],
        "cash_and_cash_equivalents": ["Cash And Cash Equivalents", "Cash Financial"],
        "cash_and_short_term_investments": ["Cash Cash Equivalents And Short Term Investments"],
        "accounts_receivable": ["Accounts Receivable", "Receivables"],
        "inventory": ["Inventory"],
        "net_property_plant_equipment": ["Net PPE", "Property Plant Equipment Net"],
        "current_liabilities": ["Current Liabilities", "Total Current Liabilities"],
        "current_debt": ["Current Debt"],
        "current_debt_and_capital_lease_obligation": ["Current Debt And Capital Lease Obligation"],
        "taxes_payable": ["Taxes Payable", "Income Tax Payable"],
        "total_liabilities": ["Total Liabilities"],
        "total_liabilities_net_minority_interest": ["Total Liabilities Net Minority Interest"],
        "total_non_current_liabilities": [
            "Total Non Current Liabilities Net Minority Interest",
            "Total Non Current Liabilities",
        ],
        "long_term_debt": ["Long Term Debt"],
        "long_term_debt_and_capital_lease_obligation": ["Long Term Debt And Capital Lease Obligation"],
        "total_debt": ["Total Debt"],
        "preferred_stock": ["Preferred Stock", "Preferred Shares Number"],
        "minority_interest": ["Minority Interest"],
        "stockholders_equity": ["Stockholders Equity"],
        "common_stock_equity": ["Common Stock Equity"],
        "retained_earnings": ["Retained Earnings"],
        "working_capital": ["Working Capital"],
        "ordinary_shares_number": ["Ordinary Shares Number"],
    },
    "cash_flow_statement": {
        "operating_cash_flow": [
            "Operating Cash Flow",
            "Total Cash From Operating Activities",
        ],
        "capital_expenditure": ["Capital Expenditure", "Capital Expenditures"],
        "free_cash_flow": ["Free Cash Flow"],
        "depreciation_and_amortization": [
            "Depreciation And Amortization",
            "Depreciation Amortization Depletion",
            "Reconciled Depreciation",
        ],
        "change_in_working_capital": ["Change In Working Capital"],
        "issuance_of_capital_stock": [
            "Issuance Of Capital Stock",
            "Common Stock Issuance",
        ],
        "repurchase_of_capital_stock": [
            "Repurchase Of Capital Stock",
            "Common Stock Payments",
        ],
        "net_common_stock_issuance": ["Net Common Stock Issuance"],
        "stock_based_compensation": ["Stock Based Compensation"],
        "cash_dividends_paid": ["Cash Dividends Paid", "Common Stock Dividend Paid"],
        "issuance_of_debt": ["Issuance Of Debt"],
        "repayment_of_debt": ["Repayment Of Debt"],
    },
}

CURRENT_INFO_FIELDS = {
    "company_name": ["longName", "shortName"],
    "sector": ["sector"],
    "industry": ["industry"],
    "exchange": ["exchange"],
    "currency": ["currency", "financialCurrency"],
    "current_share_price": ["currentPrice", "regularMarketPrice"],
    "current_market_cap": ["marketCap"],
    "current_enterprise_value": ["enterpriseValue"],
    "current_shares_outstanding": ["sharesOutstanding"],
    "current_shares_short": ["sharesShort"],
    "current_short_ratio": ["shortRatio"],
    "current_short_percent_of_float": ["shortPercentOfFloat"],
    "current_insider_ownership_percent": ["heldPercentInsiders"],
    "current_institutional_ownership_percent": ["heldPercentInstitutions"],
    "current_book_value_per_share": ["bookValue"],
    "current_forward_eps": ["forwardEps"],
}



In [ ]:
def first_present(mapping, names):
    """Return the first non-missing value found under one of several keys."""
    for name in names:
        value = mapping.get(name, np.nan)
        if value is not None and not pd.isna(value):
            return value
    return np.nan


def safe_statement(ticker_object, attribute_name):
    """Read one annual statement and always return a DataFrame."""
    try:
        statement = getattr(ticker_object, attribute_name)
        if statement is None:
            return pd.DataFrame()
        return statement.copy()
    except Exception:
        return pd.DataFrame()


def find_statement_value(statement, period, yahoo_labels):
    """Find a value without replacing it with a different accounting concept."""
    if statement.empty or period not in statement.columns:
        return np.nan
    for label in yahoo_labels:
        if label in statement.index:
            value = statement.at[label, period]
            if value is not None and not pd.isna(value):
                return value
    return np.nan


def statement_periods(*statements):
    """Return the sorted union of annual fiscal-period columns."""
    periods = set()
    for statement in statements:
        if not statement.empty:
            periods.update(pd.to_datetime(statement.columns, errors="coerce"))
    return sorted((p for p in periods if not pd.isna(p)), reverse=True)


def retrieve_company(ticker_symbol):
    """Retrieve current metadata and every available annual statement period."""
    company = yf.Ticker(ticker_symbol)
    issues = []

    try:
        info = company.info or {}
    except Exception as error:
        info = {}
        issues.append(f"metadata: {type(error).__name__}: {error}")

    income = safe_statement(company, "income_stmt")
    balance = safe_statement(company, "balance_sheet")
    cash_flow = safe_statement(company, "cashflow")

    if income.empty:
        issues.append("annual income statement unavailable")
    if balance.empty:
        issues.append("annual balance sheet unavailable")
    if cash_flow.empty:
        issues.append("annual cash-flow statement unavailable")

    periods = statement_periods(income, balance, cash_flow)
    rows = []

    for period in periods:
        row = {
            "ticker": ticker_symbol,
            "fiscal_period_end": period.date().isoformat(),
            "fiscal_year": int(period.year),
        }

        for output_name, yahoo_names in CURRENT_INFO_FIELDS.items():
            row[output_name] = first_present(info, yahoo_names)

        statement_lookup = {
            "income_statement": income,
            "balance_sheet": balance,
            "cash_flow_statement": cash_flow,
        }
        for statement_name, fields in STATEMENT_FIELDS.items():
            statement = statement_lookup[statement_name]
            for output_name, yahoo_names in fields.items():
                row[output_name] = find_statement_value(statement, period, yahoo_names)

        rows.append(row)

    if not periods:
        issues.append("no annual fiscal periods returned")

    return rows, issues


def download_financial_statements(ticker_symbols, pause_seconds=0.20):
    """Download all companies while keeping a compact issue log."""
    all_rows = []
    issue_rows = []
    successful_tickers = []

    for number, ticker_symbol in enumerate(ticker_symbols, start=1):
        print(f"[{number:02d}/{len(ticker_symbols)}] {ticker_symbol}", end=" ... ")
        try:
            company_rows, company_issues = retrieve_company(ticker_symbol)
            all_rows.extend(company_rows)

            if company_rows:
                successful_tickers.append(ticker_symbol)
                print(f"{len(company_rows)} annual periods")
            else:
                print("no annual data")

            for issue in company_issues:
                issue_rows.append({"ticker": ticker_symbol, "issue": issue})
        except Exception as error:
            issue_rows.append({
                "ticker": ticker_symbol,
                "issue": f"company request failed: {type(error).__name__}: {error}",
            })
            print("request failed; continuing")

        time.sleep(pause_seconds)

    financial_data = pd.DataFrame(all_rows)
    issues = pd.DataFrame(issue_rows, columns=["ticker", "issue"])
    return financial_data, issues, successful_tickers


def download_daily_prices(ticker_symbols, years=10):
    """Use one batch request and reshape prices to ticker-by-date rows."""
    requested = list(ticker_symbols) + ["^GSPC"]
    wide_prices = yf.download(
        requested,
        period=f"{years}y",
        interval="1d",
        auto_adjust=False,
        actions=True,
        group_by="column",
        threads=True,
        progress=False,
    )

    if wide_prices.empty:
        return pd.DataFrame()

    if not isinstance(wide_prices.columns, pd.MultiIndex):
        wide_prices.columns = pd.MultiIndex.from_product([wide_prices.columns, [requested[0]]])

    tidy = wide_prices.stack(level=1, future_stack=True).reset_index()
    tidy = tidy.rename(columns={
        "Date": "date",
        "Ticker": "ticker",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adjusted_close",
        "Volume": "volume",
        "Dividends": "dividends",
        "Stock Splits": "stock_splits",
        "Capital Gains": "capital_gains",
    })
    tidy.columns.name = None
    tidy["date"] = pd.to_datetime(tidy["date"]).dt.date.astype(str)
    return tidy.sort_values(["ticker", "date"]).reset_index(drop=True)


def add_fiscal_date_prices(financial_data, daily_prices):
    """Attach the last available trading-day prices on or before each fiscal date."""
    if financial_data.empty or daily_prices.empty:
        return financial_data.copy()

    financial = financial_data.copy()
    financial["fiscal_period_end"] = pd.to_datetime(financial["fiscal_period_end"])
    price_subset = daily_prices.loc[
        daily_prices["ticker"].ne("^GSPC"),
        ["ticker", "date", "close", "adjusted_close"],
    ].copy()
    price_subset["date"] = pd.to_datetime(price_subset["date"])
    price_subset = price_subset.sort_values(["date", "ticker"])
    financial = financial.sort_values(["fiscal_period_end", "ticker"])

    merged = pd.merge_asof(
        financial,
        price_subset,
        left_on="fiscal_period_end",
        right_on="date",
        by="ticker",
        direction="backward",
        tolerance=pd.Timedelta(days=10),
    )
    merged = merged.rename(columns={
        "date": "fiscal_price_date",
        "close": "fiscal_period_end_close",
        "adjusted_close": "fiscal_period_end_adjusted_close",
    })
    merged["fiscal_period_end"] = merged["fiscal_period_end"].dt.date.astype(str)
    merged["fiscal_price_date"] = pd.to_datetime(merged["fiscal_price_date"]).dt.date.astype("string")
    return merged.sort_values(["ticker", "fiscal_period_end"], ascending=[True, False]).reset_index(drop=True)



In [ ]:
def build_acct3343_dataset(tickers):
    """Build the annual financial dataset, daily prices, and issue log."""
    financial_data, issue_log, successful_tickers = download_financial_statements(tickers)

    try:
        daily_prices = download_daily_prices(tickers, years=10)
    except Exception as error:
        daily_prices = pd.DataFrame()
        issue_log.loc[len(issue_log)] = {
            "ticker": "ALL",
            "issue": f"daily price batch failed: {type(error).__name__}: {error}",
        }

    financial_data = add_fiscal_date_prices(financial_data, daily_prices)
    return financial_data, daily_prices, issue_log


## 5. Build the dataset

This one command runs the complete process. You will see progress for all 50 companies while it runs. The three results are:

- `financial_data`: annual company and accounting observations;
- `daily_prices`: daily market prices; and
- `failures`: an issue log for unavailable statements or requests.


In [ ]:
financial_data, daily_prices, failures = build_acct3343_dataset(TICKERS)


## 6. Take a first look

Use these short pandas commands to learn the size and structure of the annual dataset. The first number in `shape` is the number of rows; the second is the number of columns.


In [ ]:
financial_data.head()


In [ ]:
financial_data.shape


In [ ]:
financial_data.columns.tolist()


In [ ]:
financial_data["ticker"].nunique()


## 7. Filter the rows

One company can have several rows—usually one for each fiscal year Yahoo returns. The first example shows only Apple. The second uses the sector reported by Yahoo and shows only Technology companies.


In [ ]:
financial_data[financial_data["ticker"] == "AAPL"]


In [ ]:
financial_data[financial_data["sector"] == "Technology"].head(10)


## 8. Missing observations are normal

Real-world financial data are incomplete. The result below shows the 15 columns with the largest share of missing values. A missing value does not necessarily mean the code failed; Yahoo may not report that accounting field for a particular company or period.


In [ ]:
financial_data.isna().mean().sort_values(ascending=False).head(15)


## 9. Save the files for later classes

Saving the results means you do not need to download everything again each time. The CSV files can be opened by pandas, Excel, and many other programs.


In [ ]:
financial_data.to_csv("acct3343_financial_data_50.csv", index=False)
daily_prices.to_csv("acct3343_daily_prices_50.csv", index=False)
failures.to_csv("acct3343_failed_tickers_50.csv", index=False)

print("Saved acct3343_financial_data_50.csv")
print("Saved acct3343_daily_prices_50.csv")
print("Saved acct3343_failed_tickers_50.csv")


### Optional: download the CSV files from Colab

Run this only when you want copies on your own computer. Your browser will download the files one at a time.


In [ ]:
from google.colab import files

files.download("acct3343_financial_data_50.csv")
files.download("acct3343_daily_prices_50.csv")
files.download("acct3343_failed_tickers_50.csv")


## 10. A quick completion check

This confirms that the fixed universe is still intact and reports the size of the data you created. Review `failures` if Yahoo logged any unavailable statements or request problems.


In [ ]:
print("Companies requested:", len(TICKERS))
print("Companies in financial data:", financial_data["ticker"].nunique())
print("Financial data shape:", financial_data.shape)
print("Daily prices shape:", daily_prices.shape)
print("Logged issues:", len(failures))

display(failures.head())


## Optional: Save your work to GitHub

Keep both your Colab notebook/code and your resulting **acct3343_financial_data_50.csv** for future classes.

- In Colab: **File → Save a copy in GitHub** to save your notebook.
- On GitHub: open your repository, choose **Add file → Upload files**, and upload the CSV.

Do not enter GitHub passwords or tokens into Python code.
